In [ ]:
import os
os.environ["AI2POT_PATH"] = "/data/home/liuhanyu/mycode/AI2Pot/"
os.environ["AI2POT_TEST_DATA_PATH"] = os.path.join(os.environ.get("AI2POT_PATH"),
                                                   "test",
                                                   "test_data",
                                                   "XYZ")
extxyz_path: str = os.path.join(os.environ.get("AI2POT_TEST_DATA_PATH"),
                                               "11_NEP_potential_PbTe",
                                               "train_m.xyz")

from typing import List
import torch
from ai2pot.data.mlffdatamodule import ExtxyzDataModule
from ai2pot.models.potential_train import LitLinearMtp
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger

# 1. Start training a `Lienar MTP model`

In [ ]:
save_dir: str = os.path.join(os.environ.get("AI2POT_PATH"),
                                            "lightning_logs")
name: str = "linear_mtp"

max_epochs: int = 500
accelerator: str = "cpu"
devices: int = 1

trainset_path: str = extxyz_path
rcut: float = 5.0
umax_num_neigh_atoms: int = 200
pbc_xyz: List[bool] = [True, True, True]
sort: bool = True
torch_float_dtype: torch._C.dtype = torch.float64
has_virial: bool = False


mtp_level: int = 16
type_map: List[int] = [52, 82]
chebyshev_size: int = 8
rmax: float = rcut
rmin: float = 0.0
umax_num_neighs: int = umax_num_neigh_atoms
zbl_rmax: float = 0.0
zbl_rmin: float = 0.0

torch.set_num_threads(16)
torch.manual_seed(4231)

In [ ]:
tb_logger: TensorBoardLogger = TensorBoardLogger(save_dir=save_dir,
                                                 name=name)
trainer: L.Trainer = L.Trainer(max_epochs=max_epochs,
                               accelerator=accelerator,
                               devices=devices,
                               logger=tb_logger,
                               limit_val_batches=0)
datamodule: ExtxyzDataModule = ExtxyzDataModule(trainset_path=trainset_path,
                                                validset_path=trainset_path,
                                                rcut=rcut,
                                                umax_num_neigh_atoms=umax_num_neigh_atoms,
                                                pbc_xyz=pbc_xyz,
                                                sort=sort,
                                                torch_float_dtype=torch_float_dtype,
                                                has_virial=has_virial)
lit_linear_mtp: LitLinearMtp = LitLinearMtp(mtp_level=mtp_level,
                                            type_map=type_map,
                                            chebyshev_size=chebyshev_size,
                                            rmax=rmax,
                                            rmin=rmin,
                                            umax_num_neighs=umax_num_neigh_atoms,
                                            fit_virial=has_virial,
                                            zbl_rmax=zbl_rmax,
                                            zbl_rmin=zbl_rmin,
                                            zbl_cks_list=None,
                                            zbl_dks_list=None,
                                            torch_float_dtype=torch_float_dtype,
                                            lr_start=1e-1,
                                            lr_end=1e-3)

trainer.fit(model=lit_linear_mtp,
            datamodule=datamodule)

# 2. Tensorboard logger

# 3. Calculate `RMSE` of `energy`, `force`, `virial`